# AgentGate reproducible DeBERTa experiment

This notebook reruns the same frozen experiment as the local project. Upload
`agentgate-deberta-source.zip`, select a CUDA GPU runtime, and run the cells.
The original run used an RTX 3060 Laptop GPU (6 GB). The frozen protocol uses
BF16: Colab L4/A100-class GPUs are suitable; a T4 runtime will fail the capability
check. Selecting a paid runtime is your choice; this notebook does not purchase
compute or connect Google Drive.

Data: 480 AI-authored synthetic examples in 60 paired families, split 288/96/96.
NotInject and deepset are separate public diagnostics. All three seeds finish
before held-out scoring. See DATA_CARD.md for provenance and limitations.

The notebook has been syntax-checked but not executed on Colab. CUDA/platform
differences may change numeric results; compare fresh runs as replications.


In [ ]:
from google.colab import files
from pathlib import Path
import zipfile

uploaded = files.upload()
archive_names = [name for name in uploaded if name.endswith('.zip')]
assert len(archive_names) == 1, 'Upload exactly the source ZIP.'
destination = Path('/content/agentgate-experiment').resolve()
destination.mkdir(exist_ok=False)
with zipfile.ZipFile(archive_names[0]) as archive:
    for name in archive.namelist():
        target = (destination / name).resolve()
        assert target.is_relative_to(destination), 'Unsafe archive path'
    archive.extractall(destination)
%cd /content/agentgate-experiment


In [ ]:
%pip install torch==2.8.0 --index-url https://download.pytorch.org/whl/cu128
%pip install -r requirements.txt


If Colab asks for a session restart after installing packages, restart before continuing, then return to `/content/agentgate-experiment`.

In [ ]:
import torch
assert torch.cuda.is_available(), 'Select a CUDA GPU runtime.'
assert torch.cuda.is_bf16_supported(), 'The frozen BF16 protocol requires an Ampere-or-newer GPU (not T4).'
print(torch.__version__, torch.cuda.get_device_name(0))


In [ ]:
%cd /content/agentgate-experiment
from pathlib import Path
import subprocess, sys
subprocess.run([sys.executable, 'fetch_sources.py'], check=True)
if not Path('data/manifest.json').exists():
    subprocess.run([sys.executable, 'prepare_data.py'], check=True)
subprocess.run([sys.executable, '-B', '-m', 'unittest', 'test_protocol.py'], check=True)


In [ ]:
# Use a new directory for every independent run. Do not retune after viewing test results.
subprocess.run([sys.executable, '-u', 'run_experiment.py', '--output', 'runs/colab-001'], check=True)
subprocess.run([sys.executable, 'summarize.py', 'runs/colab-001'], check=True)
subprocess.run([sys.executable, 'verify_artifacts.py', 'runs/colab-001'], check=True)


In [ ]:
from IPython.display import Markdown, display
display(Markdown(Path('runs/colab-001/REPORT.md').read_text()))


In [ ]:
# Download the reports and evidence. Weights remain in the runtime unless exported below.
with zipfile.ZipFile('/content/agentgate-colab-evidence.zip', 'w', zipfile.ZIP_DEFLATED) as archive:
    for path in Path('runs/colab-001').rglob('*'):
        if path.is_file() and 'checkpoint' not in path.parts and 'trainer' not in path.parts:
            archive.write(path, path.relative_to('runs/colab-001'))
files.download('/content/agentgate-colab-evidence.zip')


To retain a checkpoint before the Colab runtime expires, export a seed explicitly.
Seed 42 is the predesignated example, not a seed selected for the best test score.


In [ ]:
# Optional: weights are approximately 738 MB per seed.
# import shutil
# shutil.make_archive('/content/agentgate-seed42-checkpoint', 'zip', 'runs/colab-001/finetuned-seed42/checkpoint')
# files.download('/content/agentgate-seed42-checkpoint.zip')
